
# WIP MLflow and Transformers

| 🚨 Consider a WIP until this message goes away... 🚨 |
|-|



## 🧑‍💻 Introduction to MLflow and Transformers

[Introduction to MLflow and Transformers](https://mlflow.org/docs/latest/ml/deep-learning/transformers/tutorials/text-generation/text-generation/)

designed for beginners, focusing on machine learning workflows and model management.

[Introduction to Transformers](https://mlflow.org/docs/latest/ml/deep-learning/transformers/tutorials/text-generation/text-generation/#introduction-to-transformers):

* Transformers are a type of deep learning models for Natural Language Processing (NLP)
* [Transformers library](https://huggingface.co/docs/transformers/index) (by 🤗 Hugging Face) offers a variety of state-of-the-art pre-trained models for NLP tasks.


## Read me
[What are Hugging Face Transformers?](https://docs.databricks.com/aws/en/machine-learning/train-model/huggingface/)


## 👨‍💻 Demo: Track and Fine-Tune Models Like a Pro

Based on this [YouTube video](https://youtu.be/8qJvHSWi-hw)

[Complete notebook](https://github.com/Mayurji/Explore-Libraries/blob/main/mlflow%2Ffine-tuning-mlflow.ipynb)

In [0]:
# Disable tokenizers warnings when constructing pipelines
%env TOKENIZERS_PARALLELISM=false

import warnings

# Disable a few less-than-useful UserWarnings from setuptools and pydantic
warnings.filterwarnings("ignore", category=UserWarning)

In [0]:
# Learn more in https://docs.databricks.com/aws/en/machine-learning/ai-runtime

# Transformers 5.8.0 has a hard import of torchvision deep in its model-loading chain
# For this demo, we need to install torchvision, too (even if it's not needed explicitly)

# Learn more about the different platform versions of PyTorch
# https://docs.astral.sh/uv/guides/integration/pytorch/#using-uv-with-pytorch

%pip install --upgrade 'mlflow-skinny[databricks]>=3.12' datasets>=4.8.5 fsspec>=2026.2.0 'transformers[torch]>=5.8.1' numpy>=2.4.4 torch>=2.11.0 torchvision>=0.26.0 evaluate>=0.4.6 peft>=0.19.1
%restart_python

In [0]:
%pip show torch

In [0]:
%pip show transformers

In [0]:
%pip show datasets

In [0]:
import mlflow

help(mlflow.transformers.autolog)

In [0]:
import mlflow

print("Enable auto-logging for HuggingFace Transformers")

# known to be compatible with 4.38.2 <= transformers <= 4.57.3
mlflow.transformers.autolog()


### datasets Python Library

[https://pypi.org/project/datasets/](https://pypi.org/project/datasets/) HuggingFace community-driven open-source library of datasets


[Prepare data for fine tuning Hugging Face models](https://docs.databricks.com/aws/en/machine-learning/train-model/huggingface/load-data):

* This article demonstrates how to prepare your data for fine-tuning open source large language models with Hugging Face Transformers and Hugging Face Datasets.

In [0]:
from datasets import load_dataset_builder
from psutil._common import bytes2human

def print_dataset_size_if_provided(*args, **kwargs):
  dataset_name = args[0]
  dataset_builder = load_dataset_builder(*args, **kwargs)

  if dataset_builder.info.download_size and dataset_builder.info.dataset_size:
    print(f"{dataset_name}: download_size={bytes2human(dataset_builder.info.download_size)}, dataset_size={bytes2human(dataset_builder.info.dataset_size)}")
  else:
    print(f"Dataset size for {dataset_name} is not provided by uploader")

### Datasets Caching

[Caching for datasets](https://docs.databricks.com/aws/en/machine-learning/train-model/huggingface/load-data#caching-for-datasets)

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS jacek_mlflow;
CREATE SCHEMA IF NOT EXISTS jacek_mlflow.hf;
CREATE VOLUME IF NOT EXISTS jacek_mlflow.hf.datasets;
CREATE VOLUME IF NOT EXISTS jacek_mlflow.hf.models;

In [0]:
uc_volume_path = "/Volumes/jacek_mlflow/hf/datasets"

In [0]:
import os
os.environ["HF_HOME"] = uc_volume_path


[ucirvine/sms_spam](https://huggingface.co/datasets/ucirvine/sms_spam)

[SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection)

In [0]:
dataset_name = "ucirvine/sms_spam"


In case a direct download from Hugging Face is disabled (e.g., for security reasons), go to [ucirvine/sms_spam](https://huggingface.co/datasets/ucirvine/sms_spam/tree/main) and download the single parquet file manually.

Upload it to a Unity Catalog volume (known as `uc_volume_path`)

In [0]:
print_dataset_size_if_provided(dataset_name)

In [0]:
%sh
df -h --output=size,used,avail,pcent,target

In [0]:
import datasets
datasets.utils.logging.disable_progress_bar()

In [0]:
from datasets import get_dataset_split_names
get_dataset_split_names(dataset_name)

In [0]:
dataset_path = f"{uc_volume_path}/{dataset_name}"
base_url = f"{dataset_path}/plain_text/"

try:
    dbutils.fs.ls(base_url)
    print(f"Path exists: {base_url}")
    path_exists = True
except Exception as e:
    print(f"Path does not exist: {base_url}")
    path_exists = False

In [0]:
from datasets import load_dataset

if path_exists:
    print(f"Loading from local path: {dataset_path}")
    data_files = {"train": base_url + "train-00000-of-00001.parquet"}
    sms_spam_dataset = datasets.load_dataset("parquet", split="train", data_files=data_files)
else:
    print(f"Loading from HF Hub")

    # Set cache_dir to a Unity Catalog volume path
    # cache_dir and HF_DATASETS_CACHE won't work on serverless because /local_disk0 is read-only
    # See /databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:144
    # sms_spam_dataset = load_dataset(dataset_name, cache_dir=f"{uc_volume_path}/{dataset_name}")
    # there's only train split available
    sms_spam_dataset = datasets.load_dataset(dataset_name, split="train", cache_dir=f"{uc_volume_path}/{dataset_name}")

sms_spam_dataset

In [0]:
# Split the test dataset into train and test
sms_train_test = sms_spam_dataset.train_test_split(test_size=0.2)
train_dataset = sms_train_test["train"]
test_dataset = sms_train_test["test"]

### Tokenization

[watch this youtube video](https://youtu.be/8qJvHSWi-hw?t=165)

Hugging Face Transformers models expect tokenized input (rather than the text in the downloaded data).

Use an [AutoTokenizer](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoTokenizer) from the pre-trained base model for model compatibility.


The base model [DistilBERT base model (uncased)](https://huggingface.co/distilbert-base-uncased) is a great foundational model that is smaller and faster than [BERT base model (uncased)](https://huggingface.co/bert-base-uncased), but still provides similar behavior.

This notebook fine tunes this base model.

In [0]:
base_model = "distilbert-base-uncased"

In [0]:
# seems required to avoid transformers deps incompatibility
# FIXME Investigate when / whether it's required

from transformers import AutoModel as PreTrainedModel

In [0]:
from transformers import AutoTokenizer

# Construct a BERT tokenizer (backed by HuggingFace's tokenizers library)
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [0]:
def tokenize_function(examples):
  """Pad/truncate each text to 128 tokens.
  Enforcing the same shape could make the training faster."""
  return tokenizer(
      examples["sms"],
      padding="max_length",
      truncation=True,
      max_length=128,
  )

In [0]:
seed = 22

# Tokenize the train and test datasets
train_tokenized = train_dataset.map(tokenize_function)
train_tokenized = train_tokenized.remove_columns(["sms"]).shuffle(seed=seed)

test_tokenized = test_dataset.map(tokenize_function)
test_tokenized = test_tokenized.remove_columns(["sms"]).shuffle(seed=seed)


### Label Mapping and Model Initialization

Review [this Databricks article](https://docs.databricks.com/aws/en/machine-learning/train-model/huggingface/fine-tune-model)

In [0]:
from transformers import AutoModelForSequenceClassification

help(AutoModelForSequenceClassification.from_pretrained)


### Load Base Model

[distilbert/distilbert-base-uncased](https://huggingface.co/distilbert/distilbert-base-uncased)

[AutoModelForSequenceClassification.from_pretrained](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModelForSequenceClassification.from_pretrained) requires one of the following:

* Path to a specific file (`model.safetensors`)
* A **hub model id** of a pretrained model hosted inside a model repo on huggingface.co
* A path to a **directory** with model weights (with `config.json`)

`from_pretrained()` loads the model weights.

In [0]:
base_model_path = "/Volumes/jacek_mlflow/hf/models/distilbert-base-uncased"

In [0]:
if path_exists:
    files = dbutils.fs.ls(base_model_path)
    assert set(f.name for f in files) == {"config.json", "model.safetensors"}, "Expected config.json and model.safetensors"

In [0]:
# Set the mapping between int label and its meaning.
id2label = {0: "ham", 1: "spam"}
label2id = {"ham": 0, "spam": 1}

# Acquire the model from the Hugging Face Hub or an UC volume (when direct download is not permitted),
# providing label and id mappings so that both we and the model can 'speak' the same language.

from transformers import AutoConfig, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
  base_model_path if path_exists else "distilbert-base-uncased",
  num_labels=len(label2id),
  label2id=label2id,
  id2label=id2label,
)

### Setting up Evaluation Metrics

In [0]:
import evaluate
import numpy as np

# Define the target optimization metric
metric = evaluate.load("accuracy")

# Define a function for calculating our defined target optimization metric during training
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return metric.compute(predictions=predictions, references=labels)


### Configuring the training environment

In [0]:
from transformers import (
  Trainer,
  TrainingArguments,
)

# Checkpoints will be output to this `training_output_dir`.
training_output_dir = "/Volumes/jacek_mlflow/hf/models"
training_args = TrainingArguments(
  output_dir=training_output_dir,
  eval_strategy='epoch',
  per_device_train_batch_size=8,
  per_device_eval_batch_size=8,
  logging_steps=8,
  num_train_epochs=3,
)

# Instantiate a `Trainer` instance that will be used to initiate a training run.
trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=train_tokenized,
  eval_dataset=test_tokenized,
  compute_metrics=compute_metrics,
)


### (optional) Setting the tracking URI

In [0]:
%skip

# It's not needed in Databricks where MLflow is already configured (managed).

import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

### Set Up MLflow

In [0]:
import mlflow

is_databricks_tracking_uri = mlflow.get_tracking_uri() == 'databricks'

In [0]:
current_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
current_user

In [0]:
import mlflow

# Pick a name that you like and reflects the nature of the runs that you will be recording to the experiment.
if is_databricks_tracking_uri:
    mlflow.set_experiment(f"/Users/{current_user}/Spam Classifier Fine-Tuning")
else:
    mlflow.set_experiment("Spam Classifier Fine-Tuning")

In [0]:
model_output_dir = "/Volumes/jacek_mlflow/hf/models/sms_model"
pipeline_output_dir = "/Volumes/jacek_mlflow/hf/models/sms_pipeline"
model_artifact_path = "sms_spam_model"

In [0]:
try:
    files = dbutils.fs.ls(model_output_dir)
    skip_model_training = len(files) > 0
except:
    skip_model_training = False

print(f"Can model training be skipped? {"👍 Yes" if skip_model_training else "👎 No"}")

In [0]:
from transformers import pipeline
import mlflow

with mlflow.start_run() as run:

  print("Model training can take up to 40 minutes")
  if not skip_model_training:
    trainer.train()
    trainer.save_model(model_output_dir)
    print(f"Model saved to {model_output_dir}")
  else:
    print(f"Skipping model training. Model already exists at {model_output_dir}")

  pipe = pipeline(
    "text-classification",
    model=AutoModelForSequenceClassification.from_pretrained(model_output_dir),
    batch_size=8,
    tokenizer=tokenizer,
  )
  pipe.save_pretrained(pipeline_output_dir)
  print(f"Pipeline saved to {pipeline_output_dir}")

  mlflow.transformers.log_model(
    transformers_model=pipe, 
    artifact_path=model_artifact_path, 
    input_example="Hi there!",
  )

![](./model_finetuning.png)

### Creating Pipeline with Fine-Tuned Model

In [0]:
from transformers import pipeline

# If you're going to run this on something other than a Macbook Pro, change the device to the applicable type.
# "mps" is for Apple Silicon architecture in torch.

tuned_pipeline = pipeline(
  # "text-classification" (alias: "sentiment-analysis") returns a TextClassificationPipeline
  task="text-classification",
  # The model to be used by the pipeline to make predictions
  model=trainer.model,
  batch_size=8,
  tokenizer=tokenizer,
)
tuned_pipeline


#### Batch Inference

Load the model as a UDF using MLflow's `mlflow.pyfunc.spark_udf` and use it for batch scoring.

[mlflow.pyfunc](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html)

In [0]:
run_id = run.info.run_id
model_artifact_path = model_artifact_path

logged_model = f"runs:/{run_id}/{model_artifact_path}"

# Load model as a Spark UDF. Override result_type if the model does not return double values.
sms_spam_model_udf = mlflow.pyfunc.spark_udf(spark, model_uri=logged_model, result_type='string')

In [0]:
# test = test_df.select(test_df.text, test_df.label, sms_spam_model_udf(test_df.text).alias("prediction"))
display(test_dataset)


#### Convert HF Dataset to Spark DataFrame

`test_dataset` is a HuggingFace `Dataset` object, but we want a Spark DataFrame.

Convert the HuggingFace Dataset to a Spark DataFrame first to apply the Spark UDF.

In [0]:
test_df = spark.createDataFrame(test_dataset.to_pandas())

In [0]:
test = test_df.select(test_df.sms, test_df.label, sms_spam_model_udf(test_df.sms).alias("prediction"))
display(test)

## Learn More

* [Introduction to MLflow and Transformers](https://mlflow.org/docs/latest/ml/deep-learning/transformers/tutorials/text-generation/text-generation/)

## 🛋️ To Be Reviewed

* https://huggingface.co/docs/transformers/index
* https://pypi.org/project/datasets/

## To Be Watched

* https://youtu.be/UezTglxJC88
* https://youtu.be/96jN2OCOfLs
* https://youtu.be/2ABNr-IJNsM
* https://youtu.be/wjZofJX0v4M
* https://youtu.be/vmfaDZjeB4M
* https://youtu.be/iOdFUJiB0Zc
* https://youtu.be/9vM4p9NN0Ts
* https://youtu.be/WOS5Qxpw56Q
* https://youtu.be/kwSVtQ7dziU
* https://youtu.be/kJLiOGle3Lw
* https://youtu.be/2mlqY4FNigg